# Simple Water Balance Model (SWBM)

[Input time series:]{.underline}

- Precipitation (P)

- Net radiation (R $_{net}$)

[Output time series:]{.underline}

- Soil moisture (w)

- Evapotranspiration (E)

- Runoff (Q)

## Equations
### Water balance w
$$
w_{n+\Delta t} = w_n + (P_n - E_n - Q_n)\Delta t
$$

### Runoff Q
$$
Q_n = (\frac{w_n}{c_s})^\alpha \cdot P_n
$$

### Evapotranspiration E
$$
E_n = \frac{\beta_0(\frac{w_n}{c_s})^\lambda \cdot R_n}{\lambda \rho _w}
$$

## 1. Parameters

In [10]:
import pandas as pd
import numpy as np
df_germany = pd.read_csv('/Users/konstantinczak/Desktop/Msc EnvSc/Courses/Semester 3/Earth System Modelling/Earth-System-Modelling/data/Data_swbm_Germany.csv')
df_germany.head()
ger_precip = df_germany['tp_[mm]']
ger_radiation = df_germany['snr_[MJ/m2]'] / 2.26

In [9]:
params = {
    'c_s': 420, # soil water holding capacity [mm]
    'alpha': 4, # runoff function shape
    'gamma': 0.5, # lambda
    'beta': 0.8, # maximum of ET function
    'rho': 1, # g/cm3
    'lambda': 1
}

initial_w = 0.9 * params['c_s'] # initials soil moisture [mm]

driving_data = {
    'precipitation': ger_precip,
    'radiation': ger_radiation
}

print("Model Components:")
print(f"\nParameters:")
for param, value in params.items():
    print(f"  {param}: {value}")
print(f"\nInitial soil moisture: {initial_w} mm")
print(f"  Precipitation range: {driving_data['precipitation'].min():.1f} mm to {driving_data['precipitation'].max():.1f} mm")
print(f"  Radiation range: {driving_data['radiation'].min():.1f} MJ/m2 to {driving_data['radiation'].max():.1f} MJ/m2")

Model Components:

Parameters:
  c_s: 420
  alpha: 4
  gamma: 0.5
  beta: 0.8
  rho: 1
  lambda: 1

Initial soil moisture: 378.0 mm
  Precipitation range: -0.0 mm to 57.9 mm
  Radiation range: -3.0 MJ/m2 to 18.2 MJ/m2


## 2. Functions

In [8]:
def calculate_runoff(w, P):
    Q = (w / params['c_s']) ** params['alpha'] * P
    return Q

def calculate_evapotranspiration(w, R):
    E = (params['beta']*(w/params['c_s'])**params['lambda']*R)/params['lambda']*params['rho']
    return E

def calculate_waterbalance(P, E, Q):
    w = P - Q - E
    return w